# 🧠 02. Phân Cụm Ward HAC, Tối Ưu HRP & Dự Báo Biến Động GARCH
### *Hierarchical Risk Parity (Marcos López de Prado) & Asymmetric GJR-GARCH Tail Risk*

Notebook này hiện thực hóa tầng giải thuật cốt lõi của hệ thống:
1. **Chuyển đổi ma trận tương quan thành không gian khoảng cách Euclid.**
2. **Phân cụm phân cấp Ward Hierarchical Agglomerative Clustering (HAC).**
3. **Vẽ cây Dendrogram & sắp xếp đường chéo giả (Quasi-Diagonalization).**
4. **Phân bổ vốn đệ nhị phân (Recursive Bisection HRP).**
5. **Mô hình hóa biến động GJR-GARCH(1,1) thích ứng Trailing Stop.**


In [ ]:
import os
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / "data"

# Import trực tiếp thuật toán từ engine đã đóng gói
sys.path.append(str(PROJECT_ROOT))
import hrp_garch_engine as hrp_engine

print("Đã nạp module hrp_garch_engine thành công!")


## 1. Tính Khoảng Cách Tương Quan Phi Tuyến
Áp dụng công thức khoảng cách góc:
$$d_{i,j} = \sqrt{rac{1}{2}(1 - ho_{i,j})}$$


In [ ]:
# Đọc ma trận giá và tính tương quan
close_df = pd.read_csv(DATA_DIR / "1_market_indices" / "VN30_All_Stocks_Close_Prices.csv", parse_dates=['date']).set_index('date')
returns_df = close_df.pct_change().dropna()

# Sử dụng cửa sổ gần nhất (Lookback 90 phiên)
recent_returns = returns_df.iloc[-90:]
corr = recent_returns.corr().values
tickers = list(recent_returns.columns)

# Tính ma trận khoảng cách
dist = np.sqrt(0.5 * (1.0 - np.clip(corr, -1.0, 1.0)))
np.fill_diagonal(dist, 0.0)

dist_condensed = squareform(dist)
link = linkage(dist_condensed, method='ward')

print(f"Số lượng cổ phiếu trong mô hình: {len(tickers)}")
print(f"Ma trận liên kết Ward Linkage Matrix: {link.shape}")


## 2. Trực Quan Hóa Cây Phân Cụm Dendrogram
Dendrogram phân tách các nhóm tài sản có hành vi tương đồng tự nhiên mà không cần giả định trước phân ngành kinh tế.


In [ ]:
plt.figure(figsize=(14, 7))
dend = dendrogram(link, labels=tickers, leaf_rotation=90, leaf_font_size=11, color_threshold=0.7)
plt.title("Cây Phân Cụm Thống Kê Ward HAC - Rổ Cổ Phiếu VN30", fontsize=15, fontweight='bold')
plt.xlabel("Mã Cổ Phiếu", fontsize=12)
plt.ylabel("Khoảng Cách Liên Kết Ward (Ward Distance)", fontsize=12)
plt.axhline(y=0.7, color='red', linestyle='--', label='Ngưỡng phân tách 4 cụm chính (Threshold = 0.7)')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()


## 3. Sắp Xếp Đường Chéo Giả (Quasi-Diagonalization)
Sắp xếp lại các hàng và cột của ma trận tương quan sao cho các tài sản có tương quan cao nằm cạnh nhau trên đường chéo chính.


In [ ]:
sorted_indices = hrp_engine.get_quasi_diag(link)
sorted_tickers = [tickers[i] for i in sorted_indices]
sorted_corr = recent_returns[sorted_tickers].corr()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

sns.heatmap(recent_returns.corr(), ax=ax1, cmap='viridis', cbar=False, square=True)
ax1.set_title("Ma Trận Tương Quan Gốc (Chưa Sắp Xếp)", fontsize=13, fontweight='bold')

sns.heatmap(sorted_corr, ax=ax2, cmap='viridis', cbar=True, square=True)
ax2.set_title("Ma Trận Sau Khi Sắp Xếp Quasi-Diagonal", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()


## 4. Tính Toán Tỷ Trọng Phân Bổ HRP (Hierarchical Risk Parity)
Phân bổ vốn đệ nhị phân dựa trên nghịch đảo phương sai cụm (Cluster Variance), triệt tiêu hoàn toàn rủi ro đảo ma trận nghịch đảo gặp phải trong Markowitz MVO.


In [ ]:
cov_matrix = recent_returns.cov().values
weights_hrp = hrp_engine.get_rec_bisection(cov_matrix, sorted_indices)

weights_series = pd.Series(weights_hrp, index=tickers).sort_values(ascending=False)

plt.figure(figsize=(14, 6))
bars = plt.bar(weights_series.index, weights_series.values * 100, color='#2ca02c', edgecolor='black', alpha=0.85)
plt.title("Tỷ Trọng Tối Ưu HRP Cho Rổ VN30 (Dựa trên 90 phiên gần nhất)", fontsize=14, fontweight='bold')
plt.ylabel("Tỷ Trọng Khuyến Nghị (%)", fontsize=12)
plt.xticks(rotation=90, fontsize=10)
plt.axhline(y=(1.0 / len(tickers)) * 100, color='red', linestyle='--', label=f'Equal Weight Mặc Định ({100/len(tickers):.2f}%)')
plt.legend()
plt.tight_layout()
plt.show()

print("Top 5 Cổ Phiếu Được Phân Bổ Tỷ Trọng Cao Nhất:")
for ticker, w in weights_series.head(5).items():
    print(f"  - {ticker}: {w*100:.2f}%")


## 5. Mô Phỏng Dự Báo Biến Động GJR-GARCH(1,1) Thích Ứng
Khi phương sai dự báo bùng nổ, hệ thống tự động siết Trailing Stop từ $2.0 	imes ATR$ xuống $1.4 	imes ATR$.


In [ ]:
# Thử nghiệm tính toán GARCH volatility cho cổ phiếu đại diện (VCB hoặc HPG)
sample_ticker = 'HPG'
sample_ret = returns_df[sample_ticker] * 100 # Chuyển sang phần trăm

vol_garch = hrp_engine.fit_garch_volatility(sample_ret.iloc[-252:])
hist_vol = sample_ret.iloc[-252:].std()

print(f"Cổ phiếu: {sample_ticker}")
print(f"Biến động lịch sử (Historical Volatility 1Y): {hist_vol:.3f}%/ngày")
print(f"Biến động dự báo GJR-GARCH(1,1) ngày mai:     {vol_garch:.3f}%/ngày")
print(f"Hệ số rủi ro biến động (Vol Ratio):          {vol_garch/hist_vol:.2f}x")

if vol_garch / hist_vol > 1.25:
    print("=> TÍN HIỆU PHÒNG THỦ: Kích hoạt siết Trailing Stop về 1.4x ATR!")
else:
    print("=> TRẠNG THÁI BÌNH THƯỜNG: Duy trì Trailing Stop tiêu chuẩn 2.0x ATR.")
